In [31]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

In [32]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y = train_df['Transported']

train_df = train_df.drop('Transported', axis=1)

dataset_df = pd.concat([train_df, test_df], axis=0, sort=False)

spend_features = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
dataset_df['Total_Spend'] = dataset_df[spend_features].sum(axis=1)




In [33]:
spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for col in spend_features:
    condition1 = ((dataset_df['CryoSleep'] == True) & (dataset_df[col].isna()))
    dataset_df.loc[condition1 , col] = 0

    condition1 = ((dataset_df['CryoSleep'] == False) & (dataset_df[col].isna()))
    dataset_df.loc[condition1 , col] = dataset_df[col].median()

for col in spend_features:
    dataset_df[col] = dataset_df[col].fillna(dataset_df[col].median())


is_awake_by_spending = (dataset_df[spend_features] > 0).any(axis=1)
dataset_df.loc[is_awake_by_spending & dataset_df['CryoSleep'].isnull() , 'CryoSleep'] = False

dataset_df['CryoSleep'] = dataset_df['CryoSleep'].fillna(True)


dataset_df['Group'] = dataset_df['PassengerId'].apply(lambda x: x.split('_')[0])

mapHomePlanet = dataset_df.groupby('Group')['HomePlanet'].first().to_dict()
dataset_df['HomePlanet'] = dataset_df['HomePlanet'].fillna(dataset_df['Group'].map(mapHomePlanet))
dataset_df['HomePlanet'] = dataset_df['HomePlanet'].fillna(dataset_df['HomePlanet'].mode()[0])

dataset_df = dataset_df.drop(['Name'] , axis = 1)
dataset_df = dataset_df.drop(['PassengerId'] , axis = 1)
dataset_df = dataset_df.drop(['Group'] , axis = 1)

dataset_df['Destination'] = dataset_df['Destination'].fillna(dataset_df['Destination'].mode()[0])
dataset_df['Age'] = dataset_df['Age'].fillna(dataset_df['Age'].median())

dataset_df['Cabin'] = dataset_df['Cabin'].fillna('U/U/U')

dataset_df['VIP'] = dataset_df['VIP'].fillna(False)



In [34]:
dataset_df['Total_Spend'] = dataset_df[spend_features].sum(axis=1)

dataset_df[['Deck', 'Num', 'Side']] = dataset_df['Cabin'].str.split('/', expand=True)
dataset_df.drop(columns=['Cabin'], inplace=True)

dataset_df['Num'] = dataset_df['Num'].replace('U', '-1').astype(int)

bool_cols = ['CryoSleep', 'VIP']
dataset_df[bool_cols] = dataset_df[bool_cols].astype(int)

categ_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
dataset_df = pd.get_dummies(dataset_df, columns=categ_cols, drop_first=True)


In [35]:
X = dataset_df[:len(y)]
x_test = dataset_df[len(y):]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, 
    y, 
    test_size=0.2,      
    random_state=42 
)


xgb = XGBClassifier(n_estimators = 400, learning_rate = 0.05 ,max_depth = 4, random_state=42)

# params = {
#     'n_estimators' : [200, 300, 400],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'max_depth': [4, 5, 6]
# }


# search = RandomizedSearchCV(xgb, params, cv=5, scoring='accuracy', n_iter=10, random_state=42)
# search.fit(X_train, y_train)
# print(search.best_params_)



rf = RandomForestClassifier(max_depth=7, n_estimators=400 , random_state=42)

ensemble_model = VotingClassifier(
    estimators=[
        ('Random_Forest', rf), 
        ('XGBoost', xgb)
    ], 
    voting='soft'
)

ensemble_model.fit(X_train, y_train)



ensemble_predicts = ensemble_model.predict(X_valid)
ensemble_accuracy = accuracy_score(y_valid, ensemble_predicts)


final_predictions = ensemble_model.predict(x_test)
test_original = pd.read_csv('test.csv')

submission = pd.DataFrame({
    'PassengerId' : test_original['PassengerId'],
    'Transported': final_predictions.astype(bool)
})

submission.to_csv('submission.csv', index=False)


0.7981598619896493
